# Cloudflare Tunnel Origin Isolation Test

This notebook intentionally does **not** install ComfyUI, H3, Director, or any models.

It proves only one path:

`https://comfy.zetbros.com` → Cloudflare Tunnel → `http://127.0.0.1:8188` → tiny Python HTTP server

If the public hostname still returns **502** while the local test returns **200**, the problem is Cloudflare hostname/DNS/tunnel routing rather than ComfyUI.


## 1. Load the tunnel token from Colab Secrets

Create/keep the Colab secret **CF_TUNNEL_TOKEN**. It may contain either the raw `eyJ...` token or the complete Cloudflare install command; this cell extracts only the token.


In [ ]:
from google.colab import userdata
import os, re

raw = userdata.get('CF_TUNNEL_TOKEN')
if not raw:
    raise RuntimeError('CF_TUNNEL_TOKEN is missing or notebook access is disabled.')

m = re.search(r'(eyJ[A-Za-z0-9._-]+)', raw.strip())
if not m:
    raise RuntimeError('Could not find an eyJ... Cloudflare Tunnel token in CF_TUNNEL_TOKEN.')

os.environ['CF_TUNNEL_TOKEN_RAW'] = m.group(1)
print('✅ Tunnel token found. Length:', len(m.group(1)))
print('Token is intentionally not printed.')


## 2. Install cloudflared

In [ ]:
import os, platform, subprocess

arch = platform.machine().lower()
cf_arch = 'amd64' if arch in ('x86_64','amd64') else 'arm64' if arch in ('aarch64','arm64') else None
if not cf_arch:
    raise RuntimeError(f'Unsupported architecture: {arch}')

if subprocess.run(['bash','-lc','command -v cloudflared >/dev/null 2>&1']).returncode != 0:
    url = f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}'
    subprocess.run(['curl','-fL','--retry','3','--retry-delay','2',url,'-o','/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod','0755','/usr/local/bin/cloudflared'], check=True)

print(subprocess.check_output(['cloudflared','--version'], text=True).strip())


## 3. Start a tiny origin server on 127.0.0.1:8188

This deliberately replaces anything currently using port 8188 in this test runtime.


In [ ]:
import os, subprocess, textwrap, time, signal
from pathlib import Path

LOG_DIR = Path('/content/cf-origin-test')
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Stop previous test processes only.
subprocess.run("pkill -f 'cf_origin_test_server.py'", shell=True, check=False)
subprocess.run("pkill -f 'cloudflared.*tunnel.*run'", shell=True, check=False)

server_py = LOG_DIR / 'cf_origin_test_server.py'
server_py.write_text(r'''
from http.server import BaseHTTPRequestHandler, HTTPServer
from datetime import datetime, timezone

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        print(f"REQUEST {datetime.now(timezone.utc).isoformat()} path={self.path} host={self.headers.get('Host')}", flush=True)
        body = b'CLOUDFLARE TUNNEL ORIGIN TEST OK\n'
        self.send_response(200)
        self.send_header('Content-Type', 'text/plain; charset=utf-8')
        self.send_header('Content-Length', str(len(body)))
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, fmt, *args):
        print('HTTP ' + (fmt % args), flush=True)

HTTPServer(('127.0.0.1', 8188), Handler).serve_forever()
''')

origin_log = open(LOG_DIR/'origin.log','w')
origin = subprocess.Popen(['python', str(server_py)], stdout=origin_log, stderr=subprocess.STDOUT)
(LOG_DIR/'origin.pid').write_text(str(origin.pid))
time.sleep(1)

print('Origin PID:', origin.pid)
subprocess.run(['ss','-ltnp'], check=False)


## 4. Prove the local origin works

In [ ]:
import subprocess

print(subprocess.check_output(['curl','-sv','--max-time','5','http://127.0.0.1:8188/'], stderr=subprocess.STDOUT, text=True))


## 5. Start the named Cloudflare Tunnel

The tunnel's Published Application in Cloudflare must be:

- Hostname: `comfy.zetbros.com`
- Service: `http://127.0.0.1:8188`


In [ ]:
import os, subprocess, time
from pathlib import Path

LOG_DIR = Path('/content/cf-origin-test')
cf_log = open(LOG_DIR/'cloudflared.log','w')
cmd = [
    'cloudflared','tunnel','--no-autoupdate','--loglevel','debug','run',
    '--token', os.environ['CF_TUNNEL_TOKEN_RAW']
]
cf = subprocess.Popen(cmd, stdout=cf_log, stderr=subprocess.STDOUT)
(LOG_DIR/'cloudflared.pid').write_text(str(cf.pid))
print('cloudflared PID:', cf.pid)

for i in range(30):
    time.sleep(1)
    if cf.poll() is not None:
        raise RuntimeError('cloudflared exited early. Run the diagnostics cell below.')
    text = (LOG_DIR/'cloudflared.log').read_text(errors='replace')
    if 'Registered tunnel connection' in text:
        print('✅ At least one tunnel connection registered.')
        break
else:
    print('⚠️ No registration line seen after 30 seconds; inspect diagnostics.')


## 6. NOW test from your browser

Open **https://comfy.zetbros.com** in your normal browser.

Expected page text:

`CLOUDFLARE TUNNEL ORIGIN TEST OK`

If you still see **502 Bad Gateway**, refresh once, then immediately run the diagnostics cell below.


## 7. Diagnostics after refreshing the public URL

This does not print the tunnel token.


In [ ]:
from pathlib import Path
import subprocess

LOG_DIR = Path('/content/cf-origin-test')
print('=== LOCAL ORIGIN ===')
subprocess.run("curl -sS -D- --max-time 5 http://127.0.0.1:8188/ -o /tmp/cf-origin-body && cat /tmp/cf-origin-body", shell=True)
print('\n=== LISTENER ===')
subprocess.run("ss -ltnp | grep ':8188' || true", shell=True)
print('\n=== ORIGIN REQUEST LOG ===')
print((LOG_DIR/'origin.log').read_text(errors='replace')[-5000:])
print('\n=== CLOUDFLARED LOG (last 120 lines) ===')
lines = (LOG_DIR/'cloudflared.log').read_text(errors='replace').splitlines()
print('\n'.join(lines[-120:]))
print('\n=== DNS VIEW FROM COLAB ===')
subprocess.run("getent ahosts comfy.zetbros.com || true", shell=True)


## How to interpret the result

- **Public page shows OK** → Tunnel + DNS + hostname routing are correct. Any remaining issue is ComfyUI-specific.
- **Public page is 502 AND origin.log gets a request** → Cloudflare reached this origin; inspect the cloudflared error lines.
- **Public page is 502 AND origin.log shows no request and cloudflared shows no request/error at refresh time** → `comfy.zetbros.com` is almost certainly routed to a different/stale tunnel/DNS record. Check Cloudflare DNS for the `comfy` record. For this tunnel the CNAME target should correspond to the tunnel UUID shown in this notebook's cloudflared startup log.
